# PDF Ingestion to FAISS

Run cells top-to-bottom.

## Step 0: Imports and Environment
Load required packages and optional `.env` variables.

In [ ]:
from pathlib import Path
import os

# Optional: load variables from .env (for example HF_TOKEN).
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Import path changed across LangChain versions; fallback keeps notebook compatible.
try:
    from langchain_text_splitters.character import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embedding model converts text chunks into vectors.
from langchain_huggingface import HuggingFaceEmbeddings
# FAISS is the local vector database used by chatbot.ipynb retriever.
from langchain_community.vectorstores import FAISS
# PDF loader parses each PDF page into LangChain Document objects.
from langchain_community.document_loaders import PyPDFLoader

d:\Work\Chatbot-file-tuning-alzheimer-diesease\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Configure Paths and Token
Set constants and read `HF_TOKEN` if available.

In [ ]:
# Input folder for source PDFs.
DATA_PATH = 'data/'
# Output folder where FAISS index files will be written.
DB_FAISS_PATH = 'vectorstore/db_faiss'
# Optional token for gated/private Hugging Face models.
HF_TOKEN = os.getenv('HF_TOKEN')

print(f"Data path: {DATA_PATH}")
print(f"Vector DB path: {DB_FAISS_PATH}")
print("HF token loaded:", bool(HF_TOKEN))

Data path: data/
Vector DB path: vectorstore/db_faiss
HF token loaded: False


## Step 2: Find and Load PDFs
Scan `data/` for PDFs and load pages into `documents`.

In [ ]:
data_dir = Path(DATA_PATH)
# Collect PDFs in deterministic order so repeated ingests are reproducible.
pdf_paths = sorted(data_dir.glob('*.pdf'))

if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in '{data_dir}'.")

documents = []
for pdf_file_path in pdf_paths:
    # Each loader.load() call returns one Document per PDF page.
    loader = PyPDFLoader(str(pdf_file_path))
    documents.extend(loader.load())

print(f"PDF files found: {len(pdf_paths)}")
print(f"Document pages loaded: {len(documents)}")

PDF files found: 1
Document pages loaded: 214


## Step 3: Split Documents into Chunks
Split text for better retrieval quality.

In [ ]:
# Split long pages into smaller overlapping chunks for better retrieval accuracy.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    # Overlap preserves context at chunk boundaries.
    chunk_overlap=50,
    # Default separators are used to split on paragraphs/sentences when possible.
)
texts = text_splitter.split_documents(documents)

print(f"Chunks after split: {len(texts)}")

Chunks after split: 930


## Step 4: Build Embeddings
Create the embedding model used by FAISS.

In [ ]:
# Run embeddings on CPU to avoid requiring a GPU.
embedding_kwargs = {'device': 'cpu'}
if HF_TOKEN:
    embedding_kwargs['token'] = HF_TOKEN

# Must match chatbot.ipynb embedding model for consistent similarity search.
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs=embedding_kwargs,
)
print('Embeddings model ready.')

Embeddings model ready.


## Step 5: Create and Save FAISS Index
Generate vectors from chunks and save to disk.

In [ ]:
# Create vector index from chunked documents.
db = FAISS.from_documents(texts, embeddings)
# Persist index to disk so chatbot.ipynb can load it later.
db.save_local(DB_FAISS_PATH)
print(f"Saved FAISS index to: {DB_FAISS_PATH}")

Saved FAISS index to: vectorstore/db_faiss
